In [ ]:
!pip install groq -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 4.9 MB/s eta 0:00:00


In [ ]:
from groq import Groq

client = Groq(api_key="YOUR_GROQ_API_KEY_HERE")

response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": "Say hello in one sentence"}]
)

print(response.choices[0].message.content)

Hello, it's nice to meet you and I'm here to help with any questions or topics you'd like to discuss.


In [ ]:
import json

candidates_db = [
    {"name": "Rahul Sharma", "experience": 3, "skills": "Python, REST APIs, SQL, Git, Docker", "title": "Software Engineer"},
    {"name": "Priya Patel", "experience": 2, "skills": "Python, Django, PostgreSQL, Git", "title": "Backend Developer"},
    {"name": "Arjun Mehta", "experience": 4, "skills": "Python, Flask, REST APIs, MySQL, AWS", "title": "Senior Developer"},
    {"name": "Sneha Reddy", "experience": 2, "skills": "Python, FastAPI, SQL, Docker, Git", "title": "Python Developer"},
    {"name": "Vikram Singh", "experience": 5, "skills": "Python, REST APIs, PostgreSQL, Kubernetes, Git", "title": "Lead Engineer"},
    {"name": "Ananya Iyer", "experience": 1, "skills": "Python, SQL, Git, HTML, CSS", "title": "Junior Developer"},
    {"name": "Rohan Gupta", "experience": 3, "skills": "Python, Django, REST APIs, Redis, Git", "title": "Full Stack Developer"},
    {"name": "Kavya Nair", "experience": 4, "skills": "Python, Machine Learning, SQL, TensorFlow, Git", "title": "ML Engineer"},
    {"name": "Amit Joshi", "experience": 2, "skills": "Python, Flask, MySQL, Docker, Git", "title": "Backend Developer"},
    {"name": "Divya Krishnan", "experience": 3, "skills": "Python, REST APIs, MongoDB, AWS, Git", "title": "Cloud Developer"},
    {"name": "Suresh Babu", "experience": 6, "skills": "Python, Django, PostgreSQL, Docker, Kubernetes", "title": "Senior Engineer"},
    {"name": "Neha Verma", "experience": 2, "skills": "Python, FastAPI, REST APIs, SQL, Git", "title": "API Developer"},
    {"name": "Karthik Rajan", "experience": 4, "skills": "Python, Flask, REST APIs, PostgreSQL, Redis", "title": "Backend Lead"},
    {"name": "Pooja Desai", "experience": 1, "skills": "Python, SQL, Git, Django", "title": "Junior Python Developer"},
    {"name": "Manish Tiwari", "experience": 5, "skills": "Python, REST APIs, AWS, Docker, PostgreSQL", "title": "DevOps Engineer"},
]

with open("candidates.json", "w") as f:
    json.dump(candidates_db, f, indent=2)

print(f"Created candidates.json with {len(candidates_db)} candidates!")

Created candidates.json with 15 candidates!


In [ ]:
import json
import re

# Load candidates
with open("candidates.json", "r") as f:
    candidates_db = json.load(f)

# Test with just first candidate
c = candidates_db[0]
jd = "We are looking for a Python Developer with 3+ years experience. Skills: Python, REST APIs, SQL, Docker, Git."

print(f"Testing with: {c['name']}")
print(f"Profile: {c}")

# Match score
score_resp = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": f"""
    Score this candidate for the job.
    Reply in this exact format:
    Match Score: XX
    Matched Skills: skills that match
    Missing Skills: skills that are missing
    Reason: one sentence

    Job: {jd}
    Candidate: {json.dumps(c)}
    """}]
)
print("\n--- MATCH SCORING ---")
print(score_resp.choices[0].message.content)

# Conversation
conv_resp = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": f"""
    Simulate a 3 exchange recruiter-candidate conversation.
    At the end write exactly:
    Interest Score: XX

    Job: {jd}
    Candidate: {json.dumps(c)}
    """}]
)
print("\n--- CONVERSATION ---")
print(conv_resp.choices[0].message.content)

Testing with: Rahul Sharma
Profile: {'name': 'Rahul Sharma', 'experience': 3, 'skills': 'Python, REST APIs, SQL, Git, Docker', 'title': 'Software Engineer'}

--- MATCH SCORING ---
Match Score: 100
Matched Skills: Python, REST APIs, SQL, Docker, Git
Missing Skills: None
Reason: The candidate's skills and experience perfectly match the job requirements.

--- CONVERSATION ---
Exchange 1:
Recruiter: Hi Rahul, thanks for considering our company. Can you tell me a little about your background and how you think your skills align with our job requirements?
Candidate: Absolutely. With 3 years of experience as a Software Engineer, I've had the opportunity to work extensively with Python, REST APIs, SQL, and Git. I've also gained experience with Docker, which I believe is a valuable asset for this role.

Exchange 2:
Recruiter: That's great to hear, Rahul. Can you give me an example of a project you worked on that involved multiple stakeholders and a complex technical problem? How did you handle i

In [ ]:
import json
import re

# Load candidates from JSON
with open("candidates.json", "r") as f:
    candidates_db = json.load(f)

print("="*60)
print("   AI TALENT SCOUTING & ENGAGEMENT AGENT")
print("="*60)

jd = input("Paste your Job Description here: ")

# Step 1 - Find top 5 matching candidates from JSON
print("\nSearching candidates database...\n")

match_resp = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": f"""
    From this list of candidates, pick the TOP 5 most relevant for the job.
    Reply with ONLY their exact names, one per line, no numbering.

    Job: {jd}

    Candidates:
    {json.dumps(candidates_db, indent=2)}
    """}]
)

selected_names = [line.strip() for line in
                  match_resp.choices[0].message.content.strip().split("\n")
                  if line.strip()][:5]

print(f"Selected: {selected_names}")

selected = [c for c in candidates_db if c['name'] in selected_names]
if len(selected) < 3:
    selected = candidates_db[:5]

final_results = []

for c in selected:
    print(f"\nProcessing {c['name']}...")

    score_resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": f"""
        Score this candidate for the job.
        Reply in this exact format:
        Match Score: XX
        Matched Skills: skills that match
        Missing Skills: skills that are missing
        Reason: one sentence

        Job: {jd}
        Candidate: {json.dumps(c)}
        """}]
    )
    score_text = score_resp.choices[0].message.content
    match_num = re.search(r'Match Score:\s*(\d+)', score_text)
    matched = re.search(r'Matched Skills:\s*(.+)', score_text)
    missing = re.search(r'Missing Skills:\s*(.+)', score_text)
    reason = re.search(r'Reason:\s*(.+)', score_text)

    match_score = int(match_num.group(1)) if match_num else 0
    matched_skills = matched.group(1).strip() if matched else ""
    missing_skills = missing.group(1).strip() if missing else ""
    reason_text = reason.group(1).strip() if reason else ""

    conv_resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": f"""
        Simulate a 3 exchange recruiter-candidate conversation.
        At the end write exactly:
        Interest Score: XX

        Job: {jd}
        Candidate: {json.dumps(c)}
        """}]
    )
    conv_text = conv_resp.choices[0].message.content
    interest_matches = re.findall(r'Interest Score:\s*(\d+)', conv_text)
    interest_score = int(interest_matches[-1]) if interest_matches else 0

    final_score = round(0.7 * match_score + 0.3 * interest_score)

    final_results.append({
        "Name": c['name'],
        "Match": match_score,
        "Interest": interest_score,
        "Final": final_score,
        "Matched": matched_skills,
        "Missing": missing_skills,
        "Reason": reason_text,
        "Conversation": conv_text
    })

# Print conversations
print("\n" + "="*60)
print("        CANDIDATE CONVERSATIONS")
print("="*60)
for r in final_results:
    print(f"\n--- {r['Name']} ---")
    print(r['Conversation'])

# Final ranked table
final_results.sort(key=lambda x: x['Final'], reverse=True)

print("\n" + "="*80)
print("           FINAL RANKED SHORTLIST")
print("="*80)
print(f"{'Rank':<5} {'Name':<18} {'Match':<8} {'Interest':<10} {'Final':<8} {'Matched Skills':<25} Missing")
print("-"*80)
for i, r in enumerate(final_results, 1):
    print(f"{i:<5} {r['Name']:<18} {r['Match']:<8} {r['Interest']:<10} {r['Final']:<8} {r['Matched']:<25} {r['Missing']}")

   AI TALENT SCOUTING & ENGAGEMENT AGENT
Paste your Job Description here: We are looking for a Python Developer with 3 years experience. Skills required: Python, REST APIs, PostgreSQL, Docker, Git.

Searching candidates database...

Selected: ['Rahul Sharma', 'Rohan Gupta', 'Divya Krishnan', 'Vikram Singh', 'Manish Tiwari']

Processing Rahul Sharma...

Processing Vikram Singh...

Processing Rohan Gupta...

Processing Divya Krishnan...

Processing Manish Tiwari...

        CANDIDATE CONVERSATIONS

--- Rahul Sharma ---
Exchange 1:
Recruiter: Hi Rahul, thanks for applying for the Python Developer position. Can you tell me a little about your background and why you're interested in this role?
Candidate: Hi, thank you for having me. With 3 years of experience as a Software Engineer, I've developed strong skills in Python, REST APIs, and Git. I'm excited about this opportunity because it involves working with technologies I'm familiar with, and I'm looking to expand my expertise in PostgreSQ

In [ ]:
import gradio as gr
import json
import re

def run_agent(job_description):

    with open("candidates.json", "r") as f:
        candidates_db = json.load(f)

    match_resp = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": f"""
        From this list of candidates, pick the TOP 5 most relevant for the job.
        Reply with ONLY their exact names, one per line, no numbering.
        Job: {job_description}
        Candidates:
        {json.dumps(candidates_db, indent=2)}
        """}]
    )

    selected_names = [line.strip() for line in
                      match_resp.choices[0].message.content.strip().split("\n")
                      if line.strip()][:5]

    selected = [c for c in candidates_db if c['name'] in selected_names]
    if len(selected) < 3:
        selected = candidates_db[:5]

    final_results = []
    all_conversations = ""

    for c in selected:
        score_resp = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": f"""
            Score this candidate for the job.
            Reply in this exact format:
            Match Score: XX
            Matched Skills: skills that match
            Missing Skills: skills that are missing
            Reason: one sentence
            Job: {job_description}
            Candidate: {json.dumps(c)}
            """}]
        )
        score_text = score_resp.choices[0].message.content
        match_num = re.search(r'Match Score:\s*(\d+)', score_text)
        matched = re.search(r'Matched Skills:\s*(.+)', score_text)
        missing = re.search(r'Missing Skills:\s*(.+)', score_text)
        reason = re.search(r'Reason:\s*(.+)', score_text)

        match_score = int(match_num.group(1)) if match_num else 0
        matched_skills = matched.group(1).strip() if matched else ""
        missing_skills = missing.group(1).strip() if missing else ""
        reason_text = reason.group(1).strip() if reason else ""

        conv_resp = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": f"""
            Simulate a 3 exchange recruiter-candidate conversation.
            At the end write exactly:
            Interest Score: XX
            Job: {job_description}
            Candidate: {json.dumps(c)}
            """}]
        )
        conv_text = conv_resp.choices[0].message.content
        interest_matches = re.findall(r'Interest Score:\s*(\d+)', conv_text)
        interest_score = int(interest_matches[-1]) if interest_matches else 0

        final_score = round(0.7 * match_score + 0.3 * interest_score)

        final_results.append({
            "Name": c['name'],
            "Match": match_score,
            "Interest": interest_score,
            "Final": final_score,
            "Matched": matched_skills,
            "Missing": missing_skills,
            "Reason": reason_text,
        })

        all_conversations += f"\n{'='*50}\n"
        all_conversations += f"Candidate: {c['name']}\n"
        all_conversations += f"{conv_text}\n"

    final_results.sort(key=lambda x: x['Final'], reverse=True)

    table = []
    for i, r in enumerate(final_results, 1):
        table.append([i, r['Name'], r['Match'], r['Interest'], r['Final'], r['Matched'], r['Missing'], r['Reason']])

    return table, all_conversations

with gr.Blocks(title="AI Talent Scouting Agent") as app:
    gr.Markdown("# 🤖 AI Talent Scouting & Engagement Agent")
    gr.Markdown("Enter a Job Description and the agent will find, score and rank the best candidates automatically.")

    jd_input = gr.Textbox(label="Job Description", placeholder="Enter job description here...", lines=5)
    submit_btn = gr.Button("🔍 Find Candidates", variant="primary")

    gr.Markdown("## 📊 Ranked Shortlist")
    output_table = gr.Dataframe(
        headers=["Rank", "Name", "Match", "Interest", "Final Score", "Matched Skills", "Missing Skills", "Reason"],
        label="Candidate Rankings"
    )

    gr.Markdown("## 💬 Candidate Conversations")
    output_conv = gr.Textbox(label="Conversations", lines=20)

    submit_btn.click(fn=run_agent, inputs=jd_input, outputs=[output_table, output_conv])

app.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://012209a53ebc08d34d.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
